In [11]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [12]:
# PAssport Held
# To download original dataset go to - 
# https://www.ons.gov.uk/datasets/TS005/editions/2021/versions/3
passport_csv = r"C:\Users\abhimanya.achara\Downloads\TS005-2021-3-filtered-2025-02-06T17_15_47Z.csv"

## 2. Input Link to LSOA 2021 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bsc-v4-2/about

In [13]:
lsoa2021_shapefile = r"C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\UK DATA\2021 Census Data\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW.shp"

## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [14]:
threshold_value = 2.5

## 1. Import & Process Data


In [15]:
# Create Dataframe
lsoa_passport_df = pd.read_csv(passport_csv)
lsoa_passport_df.head()

,Lower layer Super Output Areas Code,Lower layer Super Output Areas,Passports held (27 categories) Code,Passports held (27 categories),Observation
0,E01000001,City of London 001A,-8,Does not apply,0
1,E01000001,City of London 001A,1,Europe: United Kingdom,1061
2,E01000001,City of London 001A,2,Europe: Ireland,26
3,E01000001,City of London 001A,3,Europe: Other Europe: EU Member countries: France,36
4,E01000001,City of London 001A,4,Europe: Other Europe: EU Member countries: Ger...,24


In [16]:
#create pivot table
lsoa_passport_df_P = pd.pivot_table(lsoa_passport_df, values='Observation', index=['Lower layer Super Output Areas Code'], columns=['Passports held (27 categories)'], aggfunc=np.sum)
lsoa_passport_df_P = lsoa_passport_df_P.reset_index()

#drop columns
lsoa_passport_df_P.drop(['Does not apply'], 1, inplace=True)

# Dictionary for renaming columns with corrected keys
column_rename_map = {
       'Europe: United Kingdom': 'united_kingdom', 
       'Europe: Ireland':'ireland',
       'Europe: Other Europe: EU Member countries: France':'france',
       'Europe: Other Europe: EU Member countries: Germany':'germany',
       'Europe: Other Europe: EU Member countries: Italy':'italy',
       'Europe: Other Europe: EU Member countries: Portugal':'portugal',
       'Europe: Other Europe: EU Member countries: Spain':'spain',
       'Europe: Other Europe: EU Member countries: Lithuania':'lithuania',
       'Europe: Other Europe: EU Member countries: Poland':'poland',
       'Europe: Other Europe: EU Member countries: Romania':'romania',
       'Europe: Other Europe: EU Member countries: Other EU countries':'europe_other',
       'Europe: Other Europe: Rest of Europe: Turkey':'turkey',
       'Europe: Other Europe: Rest of Europe: Other Europe':'rest_of_europe_other',
       'Africa: North Africa':'africa_north', 
       'Africa: Central and Western Africa':'africa_central_and_western',
       'Africa: South and Eastern Africa':'africa_south_and_eastern',
       'Middle East and Asia: Middle East':'middle_east',
       'Middle East and Asia: Eastern Asia':'eastern_asia',
       'Middle East and Asia: Southern Asia':'southern_asia',
       'Middle East and Asia: South-East Asia':'south_east_asia',
       'Middle East and Asia: Central Asia':'central_asia',
       'The Americas and the Caribbean: North America and the Caribbean':'north_american_and_caribbean',
       'The Americas and the Caribbean: Central and South America':'central_and_south_america',
       'Antarctica and Oceania, including Australasia':'antartica_oceania_and_australasia',
       'British Overseas Territories':'british_overseas_territories', 
       'No passport held': 'no passport'
}

# Rename columns using the dictionary
lsoa_passport_df_P.rename(columns=column_rename_map, inplace=True)

# Rename columns: lowercase, replace spaces with _, remove colons, and add '_count' suffix
lsoa_passport_df_P.columns = (
    lsoa_passport_df_P.columns
    .str.cat(['_count'] * len(lsoa_passport_df_P.columns))  # Append '_count' to each column
)

#rename columns
lsoa_passport_df_P.rename(columns = {'Lower layer Super Output Areas Code_count':'lsoa21cd'},inplace = True)
# Display the updated DataFrame

lsoa_passport_df_P['total_passport_pop'] = lsoa_passport_df_P.sum(axis=1,numeric_only=True)

lsoa_passport_df_P.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19876\3670169416.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa_passport_df_P.drop(['Does not apply'], 1, inplace=True)


Passports held (27 categories),lsoa21cd,africa_central_and_western_count,africa_north_count,africa_south_and_eastern_count,antartica_oceania_and_australasia_count,british_overseas_territories_count,ireland_count,france_count,germany_count,italy_count,lithuania_count,europe_other_count,poland_count,portugal_count,romania_count,spain_count,rest_of_europe_other_count,turkey_count,united_kingdom_count,central_asia_count,eastern_asia_count,middle_east_count,south_east_asia_count,southern_asia_count,no passport_count,central_and_south_america_count,north_american_and_caribbean_count,total_passport_pop
0,E01000001,0,2,3,28,0,26,36,24,34,2,66,7,7,3,10,19,10,1061,0,27,4,15,18,21,7,45,1475
1,E01000002,1,7,2,17,0,26,22,17,26,6,58,14,8,5,22,22,2,964,4,65,7,8,10,25,10,38,1386
2,E01000003,3,2,6,16,0,30,32,19,40,4,71,18,10,2,31,17,6,1143,3,36,6,6,8,51,19,31,1610
3,E01000005,10,3,4,10,0,4,14,12,15,2,33,10,19,6,51,4,0,768,2,20,7,13,26,40,12,15,1100
4,E01000006,20,0,6,2,0,11,2,9,43,87,117,6,18,152,14,10,2,1109,0,1,0,7,127,95,4,4,1846


## 2. Feature Engineering

In [17]:
# Create percentage columns for detailed ethnicity
total_passport_pop = 'total_passport_pop'
for col in lsoa_passport_df_P.columns[1:-1]:
    perc_col = col.replace('_count', '_perc')
    lsoa_passport_df_P[perc_col] = (lsoa_passport_df_P[col] / lsoa_passport_df_P[total_passport_pop]) * 100

lsoa_passport_df_P.head()

Passports held (27 categories),lsoa21cd,africa_central_and_western_count,africa_north_count,africa_south_and_eastern_count,antartica_oceania_and_australasia_count,british_overseas_territories_count,ireland_count,france_count,germany_count,italy_count,lithuania_count,europe_other_count,poland_count,portugal_count,romania_count,spain_count,rest_of_europe_other_count,turkey_count,united_kingdom_count,central_asia_count,eastern_asia_count,middle_east_count,south_east_asia_count,southern_asia_count,no passport_count,central_and_south_america_count,north_american_and_caribbean_count,total_passport_pop,africa_central_and_western_perc,africa_north_perc,africa_south_and_eastern_perc,antartica_oceania_and_australasia_perc,british_overseas_territories_perc,ireland_perc,france_perc,germany_perc,italy_perc,lithuania_perc,europe_other_perc,poland_perc,portugal_perc,romania_perc,spain_perc,rest_of_europe_other_perc,turkey_perc,united_kingdom_perc,central_asia_perc,eastern_asia_perc,middle_east_perc,south_east_asia_perc,southern_asia_perc,no passport_perc,central_and_south_america_perc,north_american_and_caribbean_perc
0,E01000001,0,2,3,28,0,26,36,24,34,2,66,7,7,3,10,19,10,1061,0,27,4,15,18,21,7,45,1475,0.000000,0.135593,0.203390,1.898305,0.0,1.762712,2.440678,1.627119,2.305085,0.135593,4.474576,0.474576,0.474576,0.203390,0.677966,1.288136,0.677966,71.932203,0.000000,1.830508,0.271186,1.016949,1.220339,1.423729,0.474576,3.050847
1,E01000002,1,7,2,17,0,26,22,17,26,6,58,14,8,5,22,22,2,964,4,65,7,8,10,25,10,38,1386,0.072150,0.505051,0.144300,1.226551,0.0,1.875902,1.587302,1.226551,1.875902,0.432900,4.184704,1.010101,0.577201,0.360750,1.587302,1.587302,0.144300,69.552670,0.288600,4.689755,0.505051,0.577201,0.721501,1.803752,0.721501,2.741703
2,E01000003,3,2,6,16,0,30,32,19,40,4,71,18,10,2,31,17,6,1143,3,36,6,6,8,51,19,31,1610,0.186335,0.124224,0.372671,0.993789,0.0,1.863354,1.987578,1.180124,2.484472,0.248447,4.409938,1.118012,0.621118,0.124224,1.925466,1.055901,0.372671,70.993789,0.186335,2.236025,0.372671,0.372671,0.496894,3.167702,1.180124,1.925466
3,E01000005,10,3,4,10,0,4,14,12,15,2,33,10,19,6,51,4,0,768,2,20,7,13,26,40,12,15,1100,0.909091,0.272727,0.363636,0.909091,0.0,0.363636,1.272727,1.090909,1.363636,0.181818,3.000000,0.909091,1.727273,0.545455,4.636364,0.363636,0.000000,69.818182,0.181818,1.818182,0.636364,1.181818,2.363636,3.636364,1.090909,1.363636
4,E01000006,20,0,6,2,0,11,2,9,43,87,117,6,18,152,14,10,2,1109,0,1,0,7,127,95,4,4,1846,1.083424,0.000000,0.325027,0.108342,0.0,0.595883,0.108342,0.487541,2.329361,4.712893,6.338028,0.325027,0.975081,8.234020,0.758397,0.541712,0.108342,60.075840,0.000000,0.054171,0.000000,0.379198,6.879740,5.146262,0.216685,0.216685


In [18]:
# Feature Engineering - Aggregating Ethnicity Counts
passport_groups = {
    "eu_europe_count": [
        'ireland_count',
        'france_count',
        'germany_count',
        'italy_count',
        'portugal_count',
        'spain_count',
        'lithuania_count',
        'poland_count',
        'romania_count',
        'europe_other_count'
    ],
    
    "rest_of_europe_count": [
        'turkey_count',
        'rest_of_europe_other_count'
    ],
    "african_count": [
       'africa_north_count', 
       'africa_central_and_western_count',
       'africa_south_and_eastern_count'
    ],
    "asia_count": [
       'eastern_asia_count',
       'southern_asia_count',
       'south_east_asia_count',
       'central_asia_count'
    ],
    "americas_and_caribbean_count": [
        'north_american_and_caribbean_count',
        'central_and_south_america_count'
    ]
}

# Compute aggregated ethnicity counts
for new_col, cols in passport_groups.items():
    lsoa_passport_df_P[new_col] = lsoa_passport_df_P[cols].sum(axis=1)

# Display results
lsoa_passport_df_P.head()

Passports held (27 categories),lsoa21cd,africa_central_and_western_count,africa_north_count,africa_south_and_eastern_count,antartica_oceania_and_australasia_count,british_overseas_territories_count,ireland_count,france_count,germany_count,italy_count,lithuania_count,europe_other_count,poland_count,portugal_count,romania_count,spain_count,rest_of_europe_other_count,turkey_count,united_kingdom_count,central_asia_count,eastern_asia_count,middle_east_count,south_east_asia_count,southern_asia_count,no passport_count,central_and_south_america_count,north_american_and_caribbean_count,total_passport_pop,africa_central_and_western_perc,africa_north_perc,africa_south_and_eastern_perc,antartica_oceania_and_australasia_perc,british_overseas_territories_perc,ireland_perc,france_perc,germany_perc,italy_perc,lithuania_perc,europe_other_perc,poland_perc,portugal_perc,romania_perc,spain_perc,rest_of_europe_other_perc,turkey_perc,united_kingdom_perc,central_asia_perc,eastern_asia_perc,middle_east_perc,south_east_asia_perc,southern_asia_perc,no passport_perc,central_and_south_america_perc,north_american_and_caribbean_perc,eu_europe_count,rest_of_europe_count,african_count,asia_count,americas_and_caribbean_count
0,E01000001,0,2,3,28,0,26,36,24,34,2,66,7,7,3,10,19,10,1061,0,27,4,15,18,21,7,45,1475,0.000000,0.135593,0.203390,1.898305,0.0,1.762712,2.440678,1.627119,2.305085,0.135593,4.474576,0.474576,0.474576,0.203390,0.677966,1.288136,0.677966,71.932203,0.000000,1.830508,0.271186,1.016949,1.220339,1.423729,0.474576,3.050847,215,29,5,60,52
1,E01000002,1,7,2,17,0,26,22,17,26,6,58,14,8,5,22,22,2,964,4,65,7,8,10,25,10,38,1386,0.072150,0.505051,0.144300,1.226551,0.0,1.875902,1.587302,1.226551,1.875902,0.432900,4.184704,1.010101,0.577201,0.360750,1.587302,1.587302,0.144300,69.552670,0.288600,4.689755,0.505051,0.577201,0.721501,1.803752,0.721501,2.741703,204,24,10,87,48
2,E01000003,3,2,6,16,0,30,32,19,40,4,71,18,10,2,31,17,6,1143,3,36,6,6,8,51,19,31,1610,0.186335,0.124224,0.372671,0.993789,0.0,1.863354,1.987578,1.180124,2.484472,0.248447,4.409938,1.118012,0.621118,0.124224,1.925466,1.055901,0.372671,70.993789,0.186335,2.236025,0.372671,0.372671,0.496894,3.167702,1.180124,1.925466,257,23,11,53,50
3,E01000005,10,3,4,10,0,4,14,12,15,2,33,10,19,6,51,4,0,768,2,20,7,13,26,40,12,15,1100,0.909091,0.272727,0.363636,0.909091,0.0,0.363636,1.272727,1.090909,1.363636,0.181818,3.000000,0.909091,1.727273,0.545455,4.636364,0.363636,0.000000,69.818182,0.181818,1.818182,0.636364,1.181818,2.363636,3.636364,1.090909,1.363636,166,4,17,61,27
4,E01000006,20,0,6,2,0,11,2,9,43,87,117,6,18,152,14,10,2,1109,0,1,0,7,127,95,4,4,1846,1.083424,0.000000,0.325027,0.108342,0.0,0.595883,0.108342,0.487541,2.329361,4.712893,6.338028,0.325027,0.975081,8.234020,0.758397,0.541712,0.108342,60.075840,0.000000,0.054171,0.000000,0.379198,6.879740,5.146262,0.216685,0.216685,459,12,26,135,8


In [19]:
# Create percentage columns for simplified ethnicity
total_passport_pop = 'total_passport_pop'
for col in lsoa_passport_df_P.columns[-5:]:
    perc_col = col.replace('_count', '_perc')
    lsoa_passport_df_P[perc_col] = (lsoa_passport_df_P[col] / lsoa_passport_df_P[total_passport_pop]) * 100

lsoa_passport_df_P.head()

Passports held (27 categories),lsoa21cd,africa_central_and_western_count,africa_north_count,africa_south_and_eastern_count,antartica_oceania_and_australasia_count,british_overseas_territories_count,ireland_count,france_count,germany_count,italy_count,lithuania_count,europe_other_count,poland_count,portugal_count,romania_count,spain_count,rest_of_europe_other_count,turkey_count,united_kingdom_count,central_asia_count,eastern_asia_count,middle_east_count,south_east_asia_count,southern_asia_count,no passport_count,central_and_south_america_count,north_american_and_caribbean_count,total_passport_pop,africa_central_and_western_perc,africa_north_perc,africa_south_and_eastern_perc,antartica_oceania_and_australasia_perc,british_overseas_territories_perc,ireland_perc,france_perc,germany_perc,italy_perc,lithuania_perc,europe_other_perc,poland_perc,portugal_perc,romania_perc,spain_perc,rest_of_europe_other_perc,turkey_perc,united_kingdom_perc,central_asia_perc,eastern_asia_perc,middle_east_perc,south_east_asia_perc,southern_asia_perc,no passport_perc,central_and_south_america_perc,north_american_and_caribbean_perc,eu_europe_count,rest_of_europe_count,african_count,asia_count,americas_and_caribbean_count,eu_europe_perc,rest_of_europe_perc,african_perc,asia_perc,americas_and_caribbean_perc
0,E01000001,0,2,3,28,0,26,36,24,34,2,66,7,7,3,10,19,10,1061,0,27,4,15,18,21,7,45,1475,0.000000,0.135593,0.203390,1.898305,0.0,1.762712,2.440678,1.627119,2.305085,0.135593,4.474576,0.474576,0.474576,0.203390,0.677966,1.288136,0.677966,71.932203,0.000000,1.830508,0.271186,1.016949,1.220339,1.423729,0.474576,3.050847,215,29,5,60,52,14.576271,1.966102,0.338983,4.067797,3.525424
1,E01000002,1,7,2,17,0,26,22,17,26,6,58,14,8,5,22,22,2,964,4,65,7,8,10,25,10,38,1386,0.072150,0.505051,0.144300,1.226551,0.0,1.875902,1.587302,1.226551,1.875902,0.432900,4.184704,1.010101,0.577201,0.360750,1.587302,1.587302,0.144300,69.552670,0.288600,4.689755,0.505051,0.577201,0.721501,1.803752,0.721501,2.741703,204,24,10,87,48,14.718615,1.731602,0.721501,6.277056,3.463203
2,E01000003,3,2,6,16,0,30,32,19,40,4,71,18,10,2,31,17,6,1143,3,36,6,6,8,51,19,31,1610,0.186335,0.124224,0.372671,0.993789,0.0,1.863354,1.987578,1.180124,2.484472,0.248447,4.409938,1.118012,0.621118,0.124224,1.925466,1.055901,0.372671,70.993789,0.186335,2.236025,0.372671,0.372671,0.496894,3.167702,1.180124,1.925466,257,23,11,53,50,15.962733,1.428571,0.683230,3.291925,3.105590
3,E01000005,10,3,4,10,0,4,14,12,15,2,33,10,19,6,51,4,0,768,2,20,7,13,26,40,12,15,1100,0.909091,0.272727,0.363636,0.909091,0.0,0.363636,1.272727,1.090909,1.363636,0.181818,3.000000,0.909091,1.727273,0.545455,4.636364,0.363636,0.000000,69.818182,0.181818,1.818182,0.636364,1.181818,2.363636,3.636364,1.090909,1.363636,166,4,17,61,27,15.090909,0.363636,1.545455,5.545455,2.454545
4,E01000006,20,0,6,2,0,11,2,9,43,87,117,6,18,152,14,10,2,1109,0,1,0,7,127,95,4,4,1846,1.083424,0.000000,0.325027,0.108342,0.0,0.595883,0.108342,0.487541,2.329361,4.712893,6.338028,0.325027,0.975081,8.234020,0.758397,0.541712,0.108342,60.075840,0.000000,0.054171,0.000000,0.379198,6.879740,5.146262,0.216685,0.216685,459,12,26,135,8,24.864572,0.650054,1.408451,7.313109,0.433369


## 4. Load GIS shapefile 

In [20]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2021boundary_df = gpd.read_file(lsoa2021_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2021boundary_df.head()

Shapefile loaded successfully from the server.


,FID,LSOA21CD,LSOA21NM,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E01000001,City of London 001A,129865.314476,2635.767993,4dd060c0-a44a-4c62-aca3-b0c88c0bf0d0,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,228419.782242,2707.816821,d0a47b5d-5649-4242-a8e7-462078975c1d,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,59054.204697,1224.573160,2904cc10-e994-4a6e-b11c-06be7fbd74d2,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,189577.709503,2275.805344,bfcf5958-f052-4388-8efd-75bb808d8b83,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,146536.995750,1966.092607,64a24862-79c1-4c8c-ab7b-d9cec1d3c0c6,"POLYGON ((545126.852 184310.838, 545145.213 18..."


## 5. GIS data management

### Remove Rename fields

In [21]:
#Drop and rename fields
lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)
lsoa2021boundary_df.rename(columns = {'LSOA21CD':'lsoa21cd','LSOA21NM':'lsoa21nm'}, inplace = True)
lsoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19876\2026090638.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)


,FID,lsoa21cd,lsoa21nm,geometry
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [22]:
# Define the file path for msoa11 to msoa21 lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\OA21_LSOA21_MSOA21_LAD22_EW_LU.csv"

# Read the Excel file
lsoalookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop oa column
lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)

#remove duplicates
lsoalookup_df = lsoalookup_df.drop_duplicates(subset='lsoa21cd', keep='first')

# Display the first few rows
lsoalookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19876\1268765203.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)


,lsoa21cd,msoa21cd,msoa21nm,lad22cd,lad22nm
0,E01000001,E02000001,City of London 001,E09000001,City of London
4,E01000003,E02000001,City of London 001,E09000001,City of London
6,E01000002,E02000001,City of London 001,E09000001,City of London
12,E01032739,E02000001,City of London 001,E09000001,City of London
13,E01032740,E02000001,City of London 001,E09000001,City of London


In [23]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoalookup_df, how = 'left', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham


#### LSOA21 to WARD22

In [24]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2022)_to_LAD_(2022)_Lookup_in_England_and_Wales_v3.csv"

# Read the Excel file
lsoawardlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)

# Rename fields
lsoawardlookup_df.rename(
    columns={
        'LSOA21CD': 'lsoa21cd',
        'WD22CD': 'wd22cd',
        'WD22NM': 'wd22nm',
        }, 
    inplace=True
)

# Display the first few rows
lsoawardlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19876\1707502962.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)


,lsoa21cd,wd22cd,wd22nm
0,E01004766,E05000650,Astley Bridge
1,E01005347,E05000722,Chadderton South
2,E01004890,E05000661,Horwich North East
3,E01004770,E05000650,Astley Bridge
4,E01004888,E05000661,Horwich North East


In [25]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoawardlookup_df, how = 'inner', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury


#### LAD22 to REGION22

In [26]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_19876\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [27]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London


### Add data management fields

In [28]:
# Add new data management fields with specified values
lsoa2021boundary_df['data_source'] = "Census 2021 (ONS Nomis - Official Census and Labour Market Statistics)"
lsoa2021boundary_df['data_resolution'] = "LSOA 2021"
lsoa2021boundary_df['data_time_period'] = pd.to_datetime("2021-02-21")  # Convert to date format
lsoa2021boundary_df['data_web_link'] = "https://www.nomisweb.co.uk/"
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/


### Update Area

In [29]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2021boundary_df['area_ha'] = lsoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2021boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700


# 7. Combine Geometry and data

In [30]:
census2021_passport_lsoa2021_gdb_df = pd.merge(lsoa2021boundary_df,lsoa_passport_df_P, how = 'left', on='lsoa21cd')
census2021_passport_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,africa_central_and_western_count,africa_north_count,africa_south_and_eastern_count,antartica_oceania_and_australasia_count,british_overseas_territories_count,ireland_count,france_count,germany_count,italy_count,lithuania_count,europe_other_count,poland_count,portugal_count,romania_count,spain_count,rest_of_europe_other_count,turkey_count,united_kingdom_count,central_asia_count,eastern_asia_count,middle_east_count,south_east_asia_count,southern_asia_count,no passport_count,central_and_south_america_count,north_american_and_caribbean_count,total_passport_pop,africa_central_and_western_perc,africa_north_perc,africa_south_and_eastern_perc,antartica_oceania_and_australasia_perc,british_overseas_territories_perc,ireland_perc,france_perc,germany_perc,italy_perc,lithuania_perc,europe_other_perc,poland_perc,portugal_perc,romania_perc,spain_perc,rest_of_europe_other_perc,turkey_perc,united_kingdom_perc,central_asia_perc,eastern_asia_perc,middle_east_perc,south_east_asia_perc,southern_asia_perc,no passport_perc,central_and_south_america_perc,north_american_and_caribbean_perc,eu_europe_count,rest_of_europe_count,african_count,asia_count,americas_and_caribbean_count,eu_europe_perc,rest_of_europe_perc,african_perc,asia_perc,americas_and_caribbean_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,0,2,3,28,0,26,36,24,34,2,66,7,7,3,10,19,10,1061,0,27,4,15,18,21,7,45,1475,0.000000,0.135593,0.203390,1.898305,0.0,1.762712,2.440678,1.627119,2.305085,0.135593,4.474576,0.474576,0.474576,0.203390,0.677966,1.288136,0.677966,71.932203,0.000000,1.830508,0.271186,1.016949,1.220339,1.423729,0.474576,3.050847,215,29,5,60,52,14.576271,1.966102,0.338983,4.067797,3.525424
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,1,7,2,17,0,26,22,17,26,6,58,14,8,5,22,22,2,964,4,65,7,8,10,25,10,38,1386,0.072150,0.505051,0.144300,1.226551,0.0,1.875902,1.587302,1.226551,1.875902,0.432900,4.184704,1.010101,0.577201,0.360750,1.587302,1.587302,0.144300,69.552670,0.288600,4.689755,0.505051,0.577201,0.721501,1.803752,0.721501,2.741703,204,24,10,87,48,14.718615,1.731602,0.721501,6.277056,3.463203
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,3,2,6,16,0,30,32,19,40,4,71,18,10,2,31,17,6,1143,3,36,6,6,8,51,19,31,1610,0.186335,0.124224,0.372671,0.993789,0.0,1.863354,1.987578,1.180124,2.484472,0.248447,4.409938,1.118012,0.621118,0.124224,1.925466,1.055901,0.372671,70.993789,0.186335,2.236025,0.372671,0.372671,0.496894,3.167702,1.180124,1.925466,257,23,11,53,50,15.962733,1.428571,0.683230,3.291925,3.105590
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,10,3,4,10,0,4,14,12,15,2,33,10,19,6,51,4,0,768,2,20,7,13,26,40,12,15,1100,0.909091,0.272727,0.363636,0.909091,0.0,0.363636,1.272727,1.090909,1.363636,0.181818,3.000000,0.909091,1.727273,0.545455,4.636364,0.363636,0.000000,69.818182,0.181818,1.818182,0.636364,1.181818,2.363636,3.636364,1.090909,1.363636,166,4,17,61,27,15.090909,0.3636

# 8. Final tweaks

In [31]:
# Reorder columns in the combined dataframe

final_column_order = ['FID',
 'lsoa21cd',
 'lsoa21nm',
 'geometry',
 'msoa21cd',
 'msoa21nm',
 'lad22cd',
 'lad22nm',
 'wd22cd',
 'wd22nm',
 'rgn22cd',
 'rgn22nm',
 'data_source',
 'data_resolution',
 'data_time_period',
 'data_web_link',
 'area_ha',
 'united_kingdom_count',
 'british_overseas_territories_count',
 'eu_europe_count',
 'rest_of_europe_count',
 'african_count',
 'asia_count',
 'americas_and_caribbean_count',
 'antartica_oceania_and_australasia_count',
 'middle_east_count',
 'no passport_count', 
 'ireland_count',
 'france_count',
 'germany_count',
 'italy_count',
 'lithuania_count', 
 'poland_count',
 'portugal_count',
 'romania_count',
 'spain_count',
 'europe_other_count',
 'turkey_count',
 'rest_of_europe_other_count',                    
 'africa_central_and_western_count',
 'africa_north_count',
 'africa_south_and_eastern_count',
 'central_asia_count',
 'eastern_asia_count',
 'south_east_asia_count',
 'southern_asia_count',
 'central_and_south_america_count',
 'north_american_and_caribbean_count', 
 'total_passport_pop',
 'united_kingdom_perc',
 'british_overseas_territories_perc',
 'eu_europe_perc',
 'rest_of_europe_perc',
 'african_perc',
 'asia_perc',
 'americas_and_caribbean_perc',
 'antartica_oceania_and_australasia_perc',
 'middle_east_perc',
 'no passport_perc', 
 'ireland_perc',
 'france_perc',
 'germany_perc',
 'italy_perc',
 'lithuania_perc', 
 'poland_perc',
 'portugal_perc',
 'romania_perc',
 'spain_perc',
 'europe_other_perc',
 'turkey_perc',
 'rest_of_europe_other_perc',                    
 'africa_central_and_western_perc',
 'africa_north_perc',
 'africa_south_and_eastern_perc',
 'central_asia_perc',
 'eastern_asia_perc',
 'south_east_asia_perc',
 'southern_asia_perc',
 'central_and_south_america_perc',
 'north_american_and_caribbean_perc'
 ]

census2021_passport_lsoa2021_gdb_df = census2021_passport_lsoa2021_gdb_df[final_column_order]

census2021_passport_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,united_kingdom_count,british_overseas_territories_count,eu_europe_count,rest_of_europe_count,african_count,asia_count,americas_and_caribbean_count,antartica_oceania_and_australasia_count,middle_east_count,no passport_count,ireland_count,france_count,germany_count,italy_count,lithuania_count,poland_count,portugal_count,romania_count,spain_count,europe_other_count,turkey_count,rest_of_europe_other_count,africa_central_and_western_count,africa_north_count,africa_south_and_eastern_count,central_asia_count,eastern_asia_count,south_east_asia_count,southern_asia_count,central_and_south_america_count,north_american_and_caribbean_count,total_passport_pop,united_kingdom_perc,british_overseas_territories_perc,eu_europe_perc,rest_of_europe_perc,african_perc,asia_perc,americas_and_caribbean_perc,antartica_oceania_and_australasia_perc,middle_east_perc,no passport_perc,ireland_perc,france_perc,germany_perc,italy_perc,lithuania_perc,poland_perc,portugal_perc,romania_perc,spain_perc,europe_other_perc,turkey_perc,rest_of_europe_other_perc,africa_central_and_western_perc,africa_north_perc,africa_south_and_eastern_perc,central_asia_perc,eastern_asia_perc,south_east_asia_perc,southern_asia_perc,central_and_south_america_perc,north_american_and_caribbean_perc
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,1061,0,215,29,5,60,52,28,4,21,26,36,24,34,2,7,7,3,10,66,10,19,0,2,3,0,27,15,18,7,45,1475,71.932203,0.0,14.576271,1.966102,0.338983,4.067797,3.525424,1.898305,0.271186,1.423729,1.762712,2.440678,1.627119,2.305085,0.135593,0.474576,0.474576,0.203390,0.677966,4.474576,0.677966,1.288136,0.000000,0.135593,0.203390,0.000000,1.830508,1.016949,1.220339,0.474576,3.050847
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,964,0,204,24,10,87,48,17,7,25,26,22,17,26,6,14,8,5,22,58,2,22,1,7,2,4,65,8,10,10,38,1386,69.552670,0.0,14.718615,1.731602,0.721501,6.277056,3.463203,1.226551,0.505051,1.803752,1.875902,1.587302,1.226551,1.875902,0.432900,1.010101,0.577201,0.360750,1.587302,4.184704,0.144300,1.587302,0.072150,0.505051,0.144300,0.288600,4.689755,0.577201,0.721501,0.721501,2.741703
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1143,0,257,23,11,53,50,16,6,51,30,32,19,40,4,18,10,2,31,71,6,17,3,2,6,3,36,6,8,19,31,1610,70.993789,0.0,15.962733,1.428571,0.683230,3.291925,3.105590,0.993789,0.372671,3.167702,1.863354,1.987578,1.180124,2.484472,0.248447,1.118012,0.621118,0.124224,1.925466,4.409938,0.372671,1.055901,0.186335,0.124224,0.372671,0.186335,2.236025,0.372671,0.496894,1.180124,1.925466
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,768,0,166,4,17,61,27,10,7,40,4,14,12,15,2,10,19,6,51,33,0,4,10,3,4,2,20,13,26,12,15,1100,69.818182,0.0,15.090909,0.363636,1.545455,5.545455,2.454545,0.909091,0.636364,3.636364,0.363636,1.272727,1.090909,1.363636,0.181818,0.909091,1.727273,0.545455,4.636364,3.000000,0.000000,0.363636,0.909091,0.272727,0.363636,0.181818,1.818182,1.1818

# 8. Create dominant field

In [32]:
def determine_dominant_group(row):
    columns = [
 'united_kingdom_perc',
 'british_overseas_territories_perc',
 'eu_europe_perc',
 'rest_of_europe_perc',
 'african_perc',
 'asia_perc',
 'americas_and_caribbean_perc',
 'antartica_oceania_and_australasia_perc',
 'middle_east_perc',
 'no passport_perc'
    ]
    
    max_value = max(row[col] for col in columns)
    threshold = max_value - (threshold_value*2)
    
    dominant_groups = [col.replace('_perc', '') for col in columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_passport_lsoa2021_gdb_df['dominant_passport_held'] = census2021_passport_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_passport_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,united_kingdom_count,british_overseas_territories_count,eu_europe_count,rest_of_europe_count,african_count,asia_count,americas_and_caribbean_count,antartica_oceania_and_australasia_count,middle_east_count,no passport_count,ireland_count,france_count,germany_count,italy_count,lithuania_count,poland_count,portugal_count,romania_count,spain_count,europe_other_count,turkey_count,rest_of_europe_other_count,africa_central_and_western_count,africa_north_count,africa_south_and_eastern_count,central_asia_count,eastern_asia_count,south_east_asia_count,southern_asia_count,central_and_south_america_count,north_american_and_caribbean_count,total_passport_pop,united_kingdom_perc,british_overseas_territories_perc,eu_europe_perc,rest_of_europe_perc,african_perc,asia_perc,americas_and_caribbean_perc,antartica_oceania_and_australasia_perc,middle_east_perc,no passport_perc,ireland_perc,france_perc,germany_perc,italy_perc,lithuania_perc,poland_perc,portugal_perc,romania_perc,spain_perc,europe_other_perc,turkey_perc,rest_of_europe_other_perc,africa_central_and_western_perc,africa_north_perc,africa_south_and_eastern_perc,central_asia_perc,eastern_asia_perc,south_east_asia_perc,southern_asia_perc,central_and_south_america_perc,north_american_and_caribbean_perc,dominant_passport_held
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,1061,0,215,29,5,60,52,28,4,21,26,36,24,34,2,7,7,3,10,66,10,19,0,2,3,0,27,15,18,7,45,1475,71.932203,0.0,14.576271,1.966102,0.338983,4.067797,3.525424,1.898305,0.271186,1.423729,1.762712,2.440678,1.627119,2.305085,0.135593,0.474576,0.474576,0.203390,0.677966,4.474576,0.677966,1.288136,0.000000,0.135593,0.203390,0.000000,1.830508,1.016949,1.220339,0.474576,3.050847,united_kingdom
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,964,0,204,24,10,87,48,17,7,25,26,22,17,26,6,14,8,5,22,58,2,22,1,7,2,4,65,8,10,10,38,1386,69.552670,0.0,14.718615,1.731602,0.721501,6.277056,3.463203,1.226551,0.505051,1.803752,1.875902,1.587302,1.226551,1.875902,0.432900,1.010101,0.577201,0.360750,1.587302,4.184704,0.144300,1.587302,0.072150,0.505051,0.144300,0.288600,4.689755,0.577201,0.721501,0.721501,2.741703,united_kingdom
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1143,0,257,23,11,53,50,16,6,51,30,32,19,40,4,18,10,2,31,71,6,17,3,2,6,3,36,6,8,19,31,1610,70.993789,0.0,15.962733,1.428571,0.683230,3.291925,3.105590,0.993789,0.372671,3.167702,1.863354,1.987578,1.180124,2.484472,0.248447,1.118012,0.621118,0.124224,1.925466,4.409938,0.372671,1.055901,0.186335,0.124224,0.372671,0.186335,2.236025,0.372671,0.496894,1.180124,1.925466,united_kingdom
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,768,0,166,4,17,61,27,10,7,40,4,14,12,15,2,10,19,6,51,33,0,4,10,3,4,2,20,13,26,12,15,1100,69.818182,0.0,15.090909,0.363636,1.545455,5.545455,2.454545,0.909091,0.636364,3.636364,0.363636,1.272727,1.090909,1.363636,0.181818,0.909091,1.727273,0.545455,4.636364,3.000000,0

In [33]:
def determine_dominant_group(row):
    columns = [
 'united_kingdom_perc',
 'british_overseas_territories_perc',
 'ireland_perc',
 'france_perc',
 'germany_perc',
 'italy_perc',
 'lithuania_perc', 
 'poland_perc',
 'portugal_perc',
 'romania_perc',
 'spain_perc',
 'europe_other_perc',
 'turkey_perc',
 'rest_of_europe_other_perc',                    
 'africa_central_and_western_perc',
 'africa_north_perc',
 'africa_south_and_eastern_perc',
 'central_asia_perc',
 'eastern_asia_perc',
 'south_east_asia_perc',
 'southern_asia_perc',
 'central_and_south_america_perc',
 'north_american_and_caribbean_perc',
 'antartica_oceania_and_australasia_perc',
 'middle_east_perc',
 'no passport_perc'
    ]
    
    max_value = max(row[col] for col in columns)
    threshold = max_value - (threshold_value)
    
    dominant_groups = [col.replace('_perc', '') for col in columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_passport_lsoa2021_gdb_df['dominant_detailed_passport_held'] = census2021_passport_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_passport_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,united_kingdom_count,british_overseas_territories_count,eu_europe_count,rest_of_europe_count,african_count,asia_count,americas_and_caribbean_count,antartica_oceania_and_australasia_count,middle_east_count,no passport_count,ireland_count,france_count,germany_count,italy_count,lithuania_count,poland_count,portugal_count,romania_count,spain_count,europe_other_count,turkey_count,rest_of_europe_other_count,africa_central_and_western_count,africa_north_count,africa_south_and_eastern_count,central_asia_count,eastern_asia_count,south_east_asia_count,southern_asia_count,central_and_south_america_count,north_american_and_caribbean_count,total_passport_pop,united_kingdom_perc,british_overseas_territories_perc,eu_europe_perc,rest_of_europe_perc,african_perc,asia_perc,americas_and_caribbean_perc,antartica_oceania_and_australasia_perc,middle_east_perc,no passport_perc,ireland_perc,france_perc,germany_perc,italy_perc,lithuania_perc,poland_perc,portugal_perc,romania_perc,spain_perc,europe_other_perc,turkey_perc,rest_of_europe_other_perc,africa_central_and_western_perc,africa_north_perc,africa_south_and_eastern_perc,central_asia_perc,eastern_asia_perc,south_east_asia_perc,southern_asia_perc,central_and_south_america_perc,north_american_and_caribbean_perc,dominant_passport_held,dominant_detailed_passport_held
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531,1061,0,215,29,5,60,52,28,4,21,26,36,24,34,2,7,7,3,10,66,10,19,0,2,3,0,27,15,18,7,45,1475,71.932203,0.0,14.576271,1.966102,0.338983,4.067797,3.525424,1.898305,0.271186,1.423729,1.762712,2.440678,1.627119,2.305085,0.135593,0.474576,0.474576,0.203390,0.677966,4.474576,0.677966,1.288136,0.000000,0.135593,0.203390,0.000000,1.830508,1.016949,1.220339,0.474576,3.050847,united_kingdom,united_kingdom
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978,964,0,204,24,10,87,48,17,7,25,26,22,17,26,6,14,8,5,22,58,2,22,1,7,2,4,65,8,10,10,38,1386,69.552670,0.0,14.718615,1.731602,0.721501,6.277056,3.463203,1.226551,0.505051,1.803752,1.875902,1.587302,1.226551,1.875902,0.432900,1.010101,0.577201,0.360750,1.587302,4.184704,0.144300,1.587302,0.072150,0.505051,0.144300,0.288600,4.689755,0.577201,0.721501,0.721501,2.741703,united_kingdom,united_kingdom
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420,1143,0,257,23,11,53,50,16,6,51,30,32,19,40,4,18,10,2,31,71,6,17,3,2,6,3,36,6,8,19,31,1610,70.993789,0.0,15.962733,1.428571,0.683230,3.291925,3.105590,0.993789,0.372671,3.167702,1.863354,1.987578,1.180124,2.484472,0.248447,1.118012,0.621118,0.124224,1.925466,4.409938,0.372671,1.055901,0.186335,0.124224,0.372671,0.186335,2.236025,0.372671,0.496894,1.180124,1.925466,united_kingdom,united_kingdom
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771,768,0,166,4,17,61,27,10,7,40,4,14,12,15,2,10,19,6,51,33,0,4,10,3,4,2,20,13,26,12,15,1100,69.818182,0.0,15.090909,0.363636,1.545455,5.545455,2.454545,0.909091,0.636364,3.636364,0.363636,1.272

# 9. Upload to geodatabase

In [34]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "census2021_lsoa2021_passport_help"  # Desired table name
primary_key_column = "lsoa21cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [35]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_passport_lsoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_passport_lsoa2021_gdb_df.set_crs(epsg=27700, inplace=True)

In [37]:
# Publish the GeoDataFrame to PostGIS
census2021_passport_lsoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'MULTIPOLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.census2021_lsoa2021_passport_help
Primary key set on column: lsoa21cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.census2021_lsoa2021_passport_help
